# Cross-Benchmark Overlap Analysis

Check whether any time series appears in multiple benchmarks. Overlapping series are not necessarily a problem (a series can genuinely be both seasonal and drifting), but it should be reported and discussed.

## Hourly Aggregation

In [1]:
import numpy as np
import pandas as pd
from itertools import combinations

benchmark_names = ["SEASON", "DRIFT", "PERIODIC_SPACES", "WORKERS", "RANDOM"]
levels = ["Institutions", "Subnets", "IPs"]

benchmarks = {}
for name in benchmark_names:
    df = pd.read_csv(f"selected_ids/hourly/{name}.csv")
    benchmarks[name] = {
        level: df[df["level"] == level]["ts_id"].tolist()
        for level in levels
    }

benchmark_names = list(benchmarks.keys())

## Per-Level Overlap Matrices

In [2]:
for level in levels:
    print(f"\n{'='*60}")
    print(f"  {level}")
    print(f"{'='*60}")
    
    # Build overlap matrix
    n = len(benchmark_names)
    matrix = pd.DataFrame(0, index=benchmark_names, columns=benchmark_names)
    
    for i, b1 in enumerate(benchmark_names):
        s1 = set(benchmarks[b1][level])
        matrix.loc[b1, b1] = len(s1)
        for j, b2 in enumerate(benchmark_names):
            if i < j:
                s2 = set(benchmarks[b2][level])
                overlap = s1 & s2
                matrix.loc[b1, b2] = len(overlap)
                matrix.loc[b2, b1] = len(overlap)
    
    display(matrix)
    
    # Print specific overlaps
    found_any = False
    for b1, b2 in combinations(benchmark_names, 2):
        s1 = set(benchmarks[b1][level])
        s2 = set(benchmarks[b2][level])
        overlap = sorted(s1 & s2)
        if overlap:
            found_any = True
            print(f"  {b1} ∩ {b2}: {overlap}")
    
    if not found_any:
        print("  No overlaps.")


  Institutions


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,5,0,0,0
DRIFT,5,25,0,0,0
PERIODIC_SPACES,0,0,0,0,0
WORKERS,0,0,0,0,0
RANDOM,0,0,0,0,13


  SEASON ∩ DRIFT: [56, 63, 70, 132, 261]

  Subnets


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,2,0,0,0
DRIFT,2,25,0,0,1
PERIODIC_SPACES,0,0,0,0,0
WORKERS,0,0,0,0,0
RANDOM,0,1,0,0,25


  SEASON ∩ DRIFT: [133, 408]
  DRIFT ∩ RANDOM: [26]

  IPs


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,0,0,0,0
DRIFT,0,25,0,0,2
PERIODIC_SPACES,0,0,25,0,0
WORKERS,0,0,0,25,0
RANDOM,0,2,0,0,25


  DRIFT ∩ RANDOM: [125740, 195100]


## Combined Overlap (All Levels Together)

In [3]:
# Combine across levels using (level, id) tuples to avoid cross-level false matches
combined = {}
for bname in benchmark_names:
    ids = set()
    for level in levels:
        for ts_id in benchmarks[bname][level]:
            ids.add((level, ts_id))
    combined[bname] = ids

print("Pairwise overlaps (level, id):")
print()

total_overlaps = 0
overlap_details = []

for b1, b2 in combinations(benchmark_names, 2):
    overlap = combined[b1] & combined[b2]
    if overlap:
        total_overlaps += len(overlap)
        items = sorted(overlap)
        print(f"{b1} ∩ {b2} ({len(overlap)}):")
        for level, ts_id in items:
            print(f"  - {level} #{ts_id}")
            overlap_details.append({"series": f"{level} #{ts_id}", "benchmark_1": b1, "benchmark_2": b2})
        print()

if total_overlaps == 0:
    print("No overlaps found across any benchmark pair.")
else:
    print(f"Total overlapping (benchmark, series) pairs: {total_overlaps}")

Pairwise overlaps (level, id):

SEASON ∩ DRIFT (7):
  - Institutions #56
  - Institutions #63
  - Institutions #70
  - Institutions #132
  - Institutions #261
  - Subnets #133
  - Subnets #408

DRIFT ∩ RANDOM (3):
  - IPs #125740
  - IPs #195100
  - Subnets #26

Total overlapping (benchmark, series) pairs: 10


## Summary Table

Which series appear in how many benchmarks?

In [4]:
# For each (level, id), count how many benchmarks it appears in
from collections import Counter

membership = {}  # (level, id) -> list of benchmark names
for bname in benchmark_names:
    for level in levels:
        for ts_id in benchmarks[bname][level]:
            key = (level, ts_id)
            membership.setdefault(key, []).append(bname)

multi = {k: v for k, v in membership.items() if len(v) > 1}

if multi:
    print("Series appearing in multiple benchmarks:\n")
    rows = []
    for (level, ts_id), bmarks in sorted(multi.items()):
        rows.append({"Level": level, "ID": ts_id, "Benchmarks": ", ".join(bmarks), "Count": len(bmarks)})
    df_multi = pd.DataFrame(rows)
    display(df_multi)
else:
    print("All selected series are unique to their benchmark — no overlaps.")

# Total unique series across all benchmarks
all_series = set()
for bname in benchmark_names:
    for level in levels:
        for ts_id in benchmarks[bname][level]:
            all_series.add((level, ts_id))

total_selected = sum(len(benchmarks[b][l]) for b in benchmark_names for l in levels)
print(f"\nTotal series slots: {total_selected}")
print(f"Unique series: {len(all_series)}")
print(f"Duplicated slots: {total_selected - len(all_series)}")

Series appearing in multiple benchmarks:



,Level,ID,Benchmarks,Count
0,IPs,125740,"DRIFT, RANDOM",2
1,IPs,195100,"DRIFT, RANDOM",2
2,Institutions,56,"SEASON, DRIFT",2
3,Institutions,63,"SEASON, DRIFT",2
4,Institutions,70,"SEASON, DRIFT",2
5,Institutions,132,"SEASON, DRIFT",2
6,Institutions,261,"SEASON, DRIFT",2
7,Subnets,26,"DRIFT, RANDOM",2
8,Subnets,133,"SEASON, DRIFT",2
9,Subnets,408,"SEASON, DRIFT",2



Total series slots: 263
Unique series: 253
Duplicated slots: 10


---

# 10-Minute Aggregation

Same overlap analysis repeated for the 10-minute resolution benchmarks.

In [5]:
benchmarks_10min = {}
for name in benchmark_names:
    df = pd.read_csv(f"selected_ids/10min/{name}.csv")
    benchmarks_10min[name] = {
        level: df[df["level"] == level]["ts_id"].tolist()
        for level in levels
    }

## Per-Level Overlap Matrices (10min)

In [6]:
for level in levels:
    print(f"\n{'='*60}")
    print(f"  {level}")
    print(f"{'='*60}")
    
    n = len(benchmark_names)
    matrix = pd.DataFrame(0, index=benchmark_names, columns=benchmark_names)
    
    for i, b1 in enumerate(benchmark_names):
        s1 = set(benchmarks_10min[b1][level])
        matrix.loc[b1, b1] = len(s1)
        for j, b2 in enumerate(benchmark_names):
            if i < j:
                s2 = set(benchmarks_10min[b2][level])
                overlap = s1 & s2
                matrix.loc[b1, b2] = len(overlap)
                matrix.loc[b2, b1] = len(overlap)
    
    display(matrix)
    
    found_any = False
    for b1, b2 in combinations(benchmark_names, 2):
        s1 = set(benchmarks_10min[b1][level])
        s2 = set(benchmarks_10min[b2][level])
        overlap = sorted(s1 & s2)
        if overlap:
            found_any = True
            print(f"  {b1} \u2229 {b2}: {overlap}")
    
    if not found_any:
        print("  No overlaps.")


  Institutions


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,3,0,0,0
DRIFT,3,25,0,0,3
PERIODIC_SPACES,0,0,0,0,0
WORKERS,0,0,0,0,0
RANDOM,0,3,0,0,19


  SEASON ∩ DRIFT: [53, 127, 186]
  DRIFT ∩ RANDOM: [92, 254, 259]

  Subnets


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,1,0,0,0
DRIFT,1,25,0,0,1
PERIODIC_SPACES,0,0,0,0,0
WORKERS,0,0,0,0,0
RANDOM,0,1,0,0,25


  SEASON ∩ DRIFT: [250]
  DRIFT ∩ RANDOM: [80]

  IPs


,SEASON,DRIFT,PERIODIC_SPACES,WORKERS,RANDOM
SEASON,25,0,0,0,0
DRIFT,0,25,0,0,4
PERIODIC_SPACES,0,0,25,0,0
WORKERS,0,0,0,25,0
RANDOM,0,4,0,0,25


  DRIFT ∩ RANDOM: [3271, 41990, 78471, 109544]


## Combined Overlap — All Levels Together (10min)

In [7]:
combined_10min = {}
for bname in benchmark_names:
    ids = set()
    for level in levels:
        for ts_id in benchmarks_10min[bname][level]:
            ids.add((level, ts_id))
    combined_10min[bname] = ids

print("Pairwise overlaps (level, id):")
print()

total_overlaps = 0
for b1, b2 in combinations(benchmark_names, 2):
    overlap = combined_10min[b1] & combined_10min[b2]
    if overlap:
        total_overlaps += len(overlap)
        items = sorted(overlap)
        print(f"{b1} \u2229 {b2} ({len(overlap)}):")
        for level, ts_id in items:
            print(f"  - {level} #{ts_id}")
        print()

if total_overlaps == 0:
    print("No overlaps found across any benchmark pair.")
else:
    print(f"Total overlapping (benchmark, series) pairs: {total_overlaps}")

Pairwise overlaps (level, id):

SEASON ∩ DRIFT (4):
  - Institutions #53
  - Institutions #127
  - Institutions #186
  - Subnets #250

DRIFT ∩ RANDOM (8):
  - IPs #3271
  - IPs #41990
  - IPs #78471
  - IPs #109544
  - Institutions #92
  - Institutions #254
  - Institutions #259
  - Subnets #80

Total overlapping (benchmark, series) pairs: 12


## Summary Table (10min)

In [8]:
membership_10min = {}
for bname in benchmark_names:
    for level in levels:
        for ts_id in benchmarks_10min[bname][level]:
            key = (level, ts_id)
            membership_10min.setdefault(key, []).append(bname)

multi_10min = {k: v for k, v in membership_10min.items() if len(v) > 1}

if multi_10min:
    print("Series appearing in multiple benchmarks:\n")
    rows = []
    for (level, ts_id), bmarks in sorted(multi_10min.items()):
        rows.append({"Level": level, "ID": ts_id, "Benchmarks": ", ".join(bmarks), "Count": len(bmarks)})
    df_multi_10min = pd.DataFrame(rows)
    display(df_multi_10min)
else:
    print("All selected series are unique to their benchmark \u2014 no overlaps.")

all_series_10min = set()
for bname in benchmark_names:
    for level in levels:
        for ts_id in benchmarks_10min[bname][level]:
            all_series_10min.add((level, ts_id))

total_selected = sum(len(benchmarks_10min[b][l]) for b in benchmark_names for l in levels)
print(f"\nTotal series slots: {total_selected}")
print(f"Unique series: {len(all_series_10min)}")
print(f"Duplicated slots: {total_selected - len(all_series_10min)}")

Series appearing in multiple benchmarks:



,Level,ID,Benchmarks,Count
0,IPs,3271,"DRIFT, RANDOM",2
1,IPs,41990,"DRIFT, RANDOM",2
2,IPs,78471,"DRIFT, RANDOM",2
3,IPs,109544,"DRIFT, RANDOM",2
4,Institutions,53,"SEASON, DRIFT",2
5,Institutions,92,"DRIFT, RANDOM",2
6,Institutions,127,"SEASON, DRIFT",2
7,Institutions,186,"SEASON, DRIFT",2
8,Institutions,254,"DRIFT, RANDOM",2
9,Institutions,259,"DRIFT, RANDOM",2



Total series slots: 269
Unique series: 257
Duplicated slots: 12


---

# Hierarchical Containment Analysis

For each benchmark, check how the selected series at one level map to entities at higher levels. For example: do the 25 selected IPs all belong to different subnets/institutions, or do they cluster inside a few?

Uses the `ids_relationship` table from the dataset (columns: `id_ip`, `id_institution_subnet`, `id_institution`).

In [9]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from utils import load_dataset

# Load any source type to access the relationship table
data = load_dataset("ips", ts_ids=0.01)  # minimal load, we only need the metadata
rel = data["dataset"].get_additional_data("ids_relationship")
print(f"Relationship table: {len(rel)} rows")
rel.head()

100%|████| 10/10 [00:08<00:00,  1.25it/s]


Config Details
    Used for database: CESNET-TimeSeries24
    Aggregation: AgreggationType.AGG_1_HOUR
    Source: SourceType.IP_ADDRESSES_SAMPLE

    Time series
        Time series IDS: [    178 1790713  137804  272968  381203  254996 1704446  336915  849787
 1792304], Length=10
    Time periods
        Train time periods: range(0, 6718)
        Val time periods: None
        Test time periods: None
        All time periods: range(0, 6718)
    Features
        Taken features: ['n_bytes']
        Default values: [0.]
        Time series ID included: True
        Time included: True    
        Time format: TimeFormat.DATETIME
    Sliding window
        Sliding window size: None
        Sliding window prediction size: None
        Sliding window step size: 1
        Set shared size: 0
    Fillers
        Filler type: NoFiller
    Transformers
        Transformer type: NoTransformer
    Anomaly handler
        Anomaly handler type: NoAnomalyHandler        
    Batch sizes
        Train 

ips (AGG_1_HOUR): 10 series loaded
Relationship table: 275124 rows


,id_ip,id_institution,id_institution_subnet
0,42,0,0
1,52,0,0
2,275,0,0
3,1026,0,0
4,1128,0,0


## Hourly Benchmarks — Containment

In [10]:
ip_to_subnet = dict(zip(rel["id_ip"], rel["id_institution_subnet"]))
ip_to_inst = dict(zip(rel["id_ip"], rel["id_institution"]))
subnet_to_inst = (
    rel.groupby("id_institution_subnet")["id_institution"]
    .first()
    .to_dict()
)

def containment_report(benchmarks_dict, label):
    print(f"\n{'#' * 60}")
    print(f"  {label}")
    print(f"{'#' * 60}")

    rows = []
    for bname, levels_dict in benchmarks_dict.items():
        ip_ids = set(levels_dict.get("IPs", []))
        subnet_ids = set(levels_dict.get("Subnets", []))
        inst_ids = set(levels_dict.get("Institutions", []))

        # --- IPs → subnets / institutions ---
        if ip_ids:
            mapped_subnets = {ip_to_subnet[i] for i in ip_ids if i in ip_to_subnet}
            mapped_insts = {ip_to_inst[i] for i in ip_ids if i in ip_to_inst}
            rows.append({
                "Benchmark": bname,
                "From": f"IPs ({len(ip_ids)})",
                "→ Subnets": len(mapped_subnets),
                "→ Institutions": len(mapped_insts),
            })

        # --- Subnets → institutions ---
        if subnet_ids:
            mapped_insts_s = {subnet_to_inst[s] for s in subnet_ids if s in subnet_to_inst}
            rows.append({
                "Benchmark": bname,
                "From": f"Subnets ({len(subnet_ids)})",
                "→ Subnets": "—",
                "→ Institutions": len(mapped_insts_s),
            })

    if rows:
        display(pd.DataFrame(rows))

containment_report(benchmarks, "Hourly")


############################################################
  Hourly
############################################################


,Benchmark,From,→ Subnets,→ Institutions
0,SEASON,IPs (25),17,15
1,SEASON,Subnets (25),—,24
2,DRIFT,IPs (25),17,15
3,DRIFT,Subnets (25),—,22
4,PERIODIC_SPACES,IPs (25),13,12
5,WORKERS,IPs (25),10,10
6,RANDOM,IPs (25),15,14
7,RANDOM,Subnets (25),—,15


## Cross-Level Overlap per Benchmark

Do any selected IPs belong to a selected subnet or institution (within the same benchmark)? Do any selected subnets belong to a selected institution?

In [11]:
def cross_level_overlap(benchmarks_dict, label):
    print(f"\n{'#' * 60}")
    print(f"  {label} — Cross-Level Containment")
    print(f"{'#' * 60}")

    for bname, levels_dict in benchmarks_dict.items():
        ip_ids = set(levels_dict.get("IPs", []))
        subnet_ids = set(levels_dict.get("Subnets", []))
        inst_ids = set(levels_dict.get("Institutions", []))

        print(f"\n--- {bname} ---")
        found = False

        # IPs whose parent subnet is also selected
        if ip_ids and subnet_ids:
            ips_in_sel_subnet = {
                i for i in ip_ids
                if ip_to_subnet.get(i) in subnet_ids
            }
            if ips_in_sel_subnet:
                found = True
                parent_subnets = {ip_to_subnet[i] for i in ips_in_sel_subnet}
                print(f"  {len(ips_in_sel_subnet)} selected IP(s) belong to {len(parent_subnets)} selected subnet(s):")
                for s in sorted(parent_subnets):
                    children = sorted(i for i in ips_in_sel_subnet if ip_to_subnet[i] == s)
                    print(f"    Subnet #{s} ← IPs {children}")

        # IPs whose parent institution is also selected
        if ip_ids and inst_ids:
            ips_in_sel_inst = {
                i for i in ip_ids
                if ip_to_inst.get(i) in inst_ids
            }
            if ips_in_sel_inst:
                found = True
                parent_insts = {ip_to_inst[i] for i in ips_in_sel_inst}
                print(f"  {len(ips_in_sel_inst)} selected IP(s) belong to {len(parent_insts)} selected institution(s):")
                for inst in sorted(parent_insts):
                    children = sorted(i for i in ips_in_sel_inst if ip_to_inst[i] == inst)
                    print(f"    Institution #{inst} ← IPs {children}")

        # Subnets whose parent institution is also selected
        if subnet_ids and inst_ids:
            subnets_in_sel_inst = {
                s for s in subnet_ids
                if subnet_to_inst.get(s) in inst_ids
            }
            if subnets_in_sel_inst:
                found = True
                parent_insts = {subnet_to_inst[s] for s in subnets_in_sel_inst}
                print(f"  {len(subnets_in_sel_inst)} selected subnet(s) belong to {len(parent_insts)} selected institution(s):")
                for inst in sorted(parent_insts):
                    children = sorted(s for s in subnets_in_sel_inst if subnet_to_inst[s] == inst)
                    print(f"    Institution #{inst} ← Subnets {children}")

        if not found:
            print("  No cross-level containment.")

cross_level_overlap(benchmarks, "Hourly")


############################################################
  Hourly — Cross-Level Containment
############################################################

--- SEASON ---
  3 selected IP(s) belong to 3 selected subnet(s):
    Subnet #91 ← IPs [132516]
    Subnet #405 ← IPs [36419]
    Subnet #530 ← IPs [114939]
  3 selected subnet(s) belong to 3 selected institution(s):
    Institution #16 ← Subnets [152]
    Institution #204 ← Subnets [451]
    Institution #252 ← Subnets [509]

--- DRIFT ---
  3 selected IP(s) belong to 2 selected subnet(s):
    Subnet #171 ← IPs [1367]
    Subnet #241 ← IPs [322201, 745827]
  3 selected IP(s) belong to 2 selected institution(s):
    Institution #26 ← IPs [1367]
    Institution #56 ← IPs [322201, 745827]
  4 selected subnet(s) belong to 4 selected institution(s):
    Institution #26 ← Subnets [171]
    Institution #56 ← Subnets [241]
    Institution #137 ← Subnets [363]
    Institution #209 ← Subnets [457]

--- PERIODIC_SPACES ---
  No cross-level 

## 10-Minute Benchmarks — Containment & Cross-Level Overlap

In [12]:
containment_report(benchmarks_10min, "10-Minute")
cross_level_overlap(benchmarks_10min, "10-Minute")


############################################################
  10-Minute
############################################################


,Benchmark,From,→ Subnets,→ Institutions
0,SEASON,IPs (25),18,15
1,SEASON,Subnets (25),—,24
2,DRIFT,IPs (25),15,13
3,DRIFT,Subnets (25),—,15
4,PERIODIC_SPACES,IPs (25),8,8
5,WORKERS,IPs (25),12,12
6,RANDOM,IPs (25),13,12
7,RANDOM,Subnets (25),—,16



############################################################
  10-Minute — Cross-Level Containment
############################################################

--- SEASON ---
  7 selected IP(s) belong to 5 selected subnet(s):
    Subnet #0 ← IPs [18214]
    Subnet #17 ← IPs [2938, 204405]
    Subnet #91 ← IPs [53602]
    Subnet #200 ← IPs [40138, 1544688]
    Subnet #241 ← IPs [435318]
  10 selected IP(s) belong to 4 selected institution(s):
    Institution #0 ← IPs [83, 15032, 18214, 38586, 78489, 747744]
    Institution #41 ← IPs [3052]
    Institution #48 ← IPs [40138, 1544688]
    Institution #175 ← IPs [4897]
  9 selected subnet(s) belong to 8 selected institution(s):
    Institution #0 ← Subnets [0]
    Institution #9 ← Subnets [136, 141]
    Institution #12 ← Subnets [148]
    Institution #48 ← Subnets [200]
    Institution #63 ← Subnets [253]
    Institution #100 ← Subnets [315]
    Institution #102 ← Subnets [318]
    Institution #127 ← Subnets [350]

--- DRIFT ---
  2 selec